In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

DB_DIR = "parllment/raw_pdf/Chroma_db/chroma_db"
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/LaBSE")

import warnings
from langchain_community.vectorstores import Chroma
from pathlib import Path

warnings.filterwarnings("ignore")

vectordb = Chroma(
    persist_directory=str(DB_DIR),
    embedding_function=embeddings,
    collection_name="2014-2018"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
sample = vectordb.get(limit=1)
if sample['metadatas']:
    print("Tárolt metaadat minta:", sample['metadatas'][0])
    print("datum_int típusa:", type(sample['metadatas'][0].get('datum_int')))
else:
    print("Az adatbázis üres!")

Tárolt metaadat minta: {'datum': '2014-05-06', 'speech_id': '4abfcbe15868', 'source': '2014-2018_2014-05-06_01.pdf', 'word_count': 37, 'id': '592146aacc47b3', 'total_chunks': 1, 'datum_int': 20140506, 'token_estimate': 78, 'chunk_seq': 0, 'part': 'Jobbik', 'breadcrumb': '2014-05-06 | Bana Tibor (Jobbik) | Napló sz.: 1', 'felszolalo': 'Bana Tibor', 'ciklus': '2014-2018', 'szam': '1', 'speech_preview': 'BANA TIBOR korjegyző: Tisztelt Országgyűlés! Tájékoztatom önöket, hogy minden képviselő és szószóló benyújtotta összeférhetetlenségi nyilatkozatát. Mi...'}
datum_int típusa: <class 'int'>


### Hasonlóság alapú lekérés (similarity serach)

In [3]:
# 2. Lekérdezés összeállítása
query = "Az MNB alapítványok körül kialakult viták összefoglalása."

# Fontos: A Chroma szigorú a típusokkal. 
# Ha a feltöltésnél a datum_int integer volt, itt is annak kell lennie.
search_filter = {
    "datum_int": {"$gte": 20150101}
}

# 3. Keresés futtatása
try:
    docs = vectordb.similarity_search(
        query, 
        k=3, 
        filter=search_filter
    )

    # 4. Eredmények megjelenítése
    if not docs:
        print("Nem található a feltételeknek megfelelő találat.")
    else:
        for doc in docs:
            # Biztonsági lekérés .get()-tel, ha esetleg hiányozna a metaadat
            breadcrumb = doc.metadata.get('breadcrumb', 'Nincs elérhető forrásadat')
            print(f"--- {breadcrumb} ---")
            print(f"Content: {doc.page_content[:200]}...\n")

except Exception as e:
    print(f"Hiba történt a lekérdezés során: {e}")

--- 2016-05-23 | Dr. Nagy Márton (Kormányzati tisztségviselő) | Napló sz.: 155 ---
Content: DR. NAGY MÁRTON, a Magyar Nemzeti Bank alelnöke: Köszönöm szépen. Tisztelt Képviselők! Én azt gondolom, hogy a felvetése első részére reagálva többször elhangzott az, hogy az Alkotmánybíróság döntése ...

--- 2016-04-26 | Dr. Tóth Bertalan (MSZP) | Napló sz.: 145 ---
Content: . De további vizsgálatokra is szükség van, ezért egy hatpontos intézkedési csomagot javasolunk az MNB alapítványainak és a költségvetés finanszírozásának átvilágítása és az alapítványok további ámokfu...

--- 2015-11-04 | Dr. Tóth Bertalan (MSZP) | Napló sz.: 113 ---
Content: . Na most, itt azért feltevődik egy kérdés, és visszatérek arra az okfejtésemre, ami a közpénz, nem közpénz vitára vonatkozik, mert ha az én állításom igaz, hogy a Magyar Nemzeti Bank által kihelyezet...



### 2. Időrendi "Idővonal" Keresés

In [4]:
query = "Úniós források elosztása"
# 2022. május 1. utáni felszólalások
search_filter = {"datum_int": {"$gt": 20150501}}

results = vectordb.similarity_search(query, k=5, filter=search_filter)

# Rendezzük sorba dátum szerint a találatokat
sorted_results = sorted(results, key=lambda x: x.metadata['datum_int'])

print(f"Időrendi találatok ({query}):\n")
for doc in sorted_results:
    m = doc.metadata
    print(f"[{m['datum']}] {m['felszolalo']} ({m['part']}):")
    print(f"   --> {doc.page_content[:150]}...\n")

Időrendi találatok (Úniós források elosztása):

[2015-05-27] Domokos László (Kormányzati tisztségviselő):
   --> . A 2016. évi költségvetési tervezés sajátossága, hogy egyszerre kell fedezetet nyújtania az Európai Unió 2007-2013-as költségvetési ciklusában EUforr...

[2015-06-09] Lázár János (Fidesz):
   --> . Szeretném azt is elmondani, hogy az Európai Unióban megosztottság van a számunkra, középeurópai fölzárkózó országok számára biztosított kohéziós ala...

[2016-05-11] Varga Mihály (Fidesz):
   --> . Tisztelt Országgyűlés! A kormány a jövőben is kiemelten kezeli az uniós támogatások felhasználását. A 2014-20-as időszak rendelkezésre álló támogatá...

[2016-11-28] Witzmann Mihály (Fidesz):
   --> . A források lehívásával kapcsolatban azonban mégis, én azt gondolom, hogy ezek a felesleges ellenzéki aggodalmak azért is érthetetlenek számomra, mer...

[2017-05-22] Lázár János (Fidesz):
   --> . Az erőforrások elosztását már itt szóba hoztam. Én tisztában vagyok azzal, hogy az eszközök 